In [32]:
from pathlib import Path
import yfinance as yf

In [33]:
RACINE = next (d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists())
DOSSIER = RACINE /"data"/ "raw" / "prix"
SYMBOLE = "NVDA"

COLONNES = ["Open" , "High", "Low", "Close", "Adj Close", "Volume", "Dividends", "Stock Splits"]

In [34]:
histo = yf.Ticker(SYMBOLE).history(period = "max" , auto_adjust =False)
histo = histo[COLONNES]

In [35]:
histo.index.name = "date"
histo["symbole"]= SYMBOLE

In [36]:
DOSSIER.mkdir(parents = True, exist_ok = True)
chemin = DOSSIER / f"{SYMBOLE}.csv"

In [37]:
histo.to_csv(chemin, encoding = "utf-8")
print(len(histo), "lignes ecrites dans" , chemin)
histo.tail(3)

6948 lignes ecrites dans C:\Users\josue\Documents\risk-modeling-project\data\raw\prix\NVDA.csv


,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,symbole
date,,,,,,,,,
2026-09-02 00:00:00-04:00,218.789993,227.949997,218.479996,224.410004,224.410004,157104700,0.0,0.0,NVDA
2026-09-03 00:00:00-04:00,226.020004,230.399994,224.750000,228.449997,228.449997,134681600,0.0,0.0,NVDA
2026-09-04 00:00:00-04:00,231.089996,234.759995,229.630005,230.360001,230.360001,134946800,0.0,0.0,NVDA


### lanncement et imporatations des base de données yahoo finance 

In [38]:
import time
import pandas as pd

In [39]:
univers = pd.read_csv(RACINE / "data" / "processed" / "univers_retenu.csv")
symboles = sorted({s for ligne in univers["symboles"] for s in ligne.split("|")})
print(len(symboles), "symboles a collecter")

113 symboles a collecter


In [40]:
echecs = []

for i, symbole in enumerate(symboles, start=1):
    chemin = DOSSIER / f"{symbole}.csv"
    if chemin.exists():
        continue
    try:
        histo = yf.Ticker(symbole).history(period="max", auto_adjust=False)
        if histo.empty:
            echecs.append((symbole, "aucune donnee"))
            continue
        histo = histo[COLONNES]
        histo.index.name = "date"
        histo["symbole"] = symbole
        histo.to_csv(chemin, encoding="utf-8")
    except Exception as erreur:
        echecs.append((symbole, f"{type(erreur).__name__} : {erreur}"))
    time.sleep(0.3)
    if i % 20 == 0:
        print(" ", i, "/", len(symboles))

print()
print("termine.", len(echecs), "echecs")
for symbole, message in echecs:
    print(" -", symbole, ":", message[:80])

$BRK.B: possibly delisted; no timezone found


  20 / 113
  40 / 113
  60 / 113
  80 / 113
  100 / 113

termine. 1 echecs
 - BRK.B : aucune donnee


In [43]:
symbole = "BRK.B"
requete = symbole.replace(".", "-")

histo = yf.Ticker(requete).history(period="max", auto_adjust=False)
histo = histo[COLONNES]
histo.index.name = "date"
histo["symbole"] = symbole
histo["symbole_yahoo"] = requete

chemin = DOSSIER / f"{requete}.csv"
histo.to_csv(chemin, encoding="utf-8")
print(len(histo), "lignes", chemin.name)

7630 lignes BRK-B.csv


### benchmark

In [44]:
BENCH = RACINE / "data" / "raw" / "benchmarks"
BENCH.mkdir(parents=True, exist_ok=True)

REFERENCES = ["SPY", "RSP", "^GSPC", "^SP500TR", "^SPXEW"]

In [45]:
for symbole in REFERENCES:
    fichier = symbole.replace("^", "") + ".csv"
    chemin = BENCH / fichier
    if chemin.exists():
        continue
    histo = yf.Ticker(symbole).history(period="max", auto_adjust=False)
    histo = histo[COLONNES]
    histo.index.name = "date"
    histo["symbole"] = symbole
    histo.to_csv(chemin, encoding="utf-8")
    print(symbole, len(histo), "lignes")
    time.sleep(0.3)

SPY 8458 lignes
RSP 5875 lignes
^GSPC 24787 lignes
^SP500TR 9742 lignes
^SPXEW 4951 lignes


In [46]:
CALENDRIER = RACINE / "data" / "raw" / "calendrier_bourse.csv"

spy = pd.read_csv(BENCH / "SPY.csv", usecols=["date"])
seances = spy["date"].str[:10]
pd.DataFrame({"date": seances}).to_csv(CALENDRIER, index=False)
print(len(seances), "seances, du", seances.iloc[0], "au", seances.iloc[-1])

nvda = pd.read_csv(DOSSIER / "NVDA.csv", usecols=["date"])["date"].str[:10]
attendues = set(seances[seances >= nvda.iloc[0]])
manquantes = sorted(attendues - set(nvda))
print(len(manquantes), "seances absentes du fichier NVDA")
print(manquantes[:10])

8458 seances, du 1993-01-29 au 2026-09-04
0 seances absentes du fichier NVDA
[]


### cas simple NVDA

In [47]:
import requests

ENTETE = {"User-Agent": "AI-Concentration-Risk-Research josuenkpoman@gmail.com"}
CIK = "0001045810"

url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK}.json"
faits = requests.get(url, headers=ENTETE, timeout=60).json()

print("sections :", list(faits["facts"].keys()))
print()
print("etiquettes dei :", list(faits["facts"]["dei"].keys()))

bloc = faits["facts"]["dei"]["EntityCommonStockSharesOutstanding"]
valeurs = bloc["units"]["shares"]
print()
print(len(valeurs), "declarations")
for v in valeurs[-3:]:
    print(v)

sections : ['dei', 'invest', 'us-gaap', 'srt', 'ecd', 'ffd']

etiquettes dei : ['EntityCommonStockSharesOutstanding', 'EntityPublicFloat']

69 declarations
{'end': '2026-02-20', 'val': 24300000000, 'accn': '0001045810-26-000021', 'fy': 2026, 'fp': 'FY', 'form': '10-K', 'filed': '2026-02-25', 'frame': 'CY2026Q1I'}
{'end': '2026-05-15', 'val': 24200000000, 'accn': '0001045810-26-000052', 'fy': 2027, 'fp': 'Q1', 'form': '10-Q', 'filed': '2026-05-20'}
{'end': '2026-08-21', 'val': 24100000000, 'accn': '0001045810-26-000075', 'fy': 2027, 'fp': 'Q2', 'form': '10-Q', 'filed': '2026-08-26', 'frame': 'CY2026Q3I'}


### cas général

In [48]:
import time

FICHIER = RACINE / "data" / "raw" / "actions_en_circulation.csv"

univers = pd.read_csv(RACINE / "data" / "processed" / "univers_retenu.csv", dtype={"cik": str})
entreprises = univers[["cik", "nom"]].drop_duplicates()

ETIQUETTES = [
    ("EntityCommonStockSharesOutstanding", "shares", "actions"),
    ("EntityPublicFloat", "USD", "flottant_usd"),
]

lignes = []
echecs = []

for i, (cik, nom) in enumerate(entreprises.itertuples(index=False), start=1):
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    try:
        dei = requests.get(url, headers=ENTETE, timeout=60).json()["facts"]["dei"]
        for etiquette, unite, notion in ETIQUETTES:
            bloc = dei.get(etiquette)
            if bloc is None:
                continue
            for v in bloc["units"].get(unite, []):
                lignes.append({"cik": cik, "nom": nom, "notion": notion,
                               "debut": v.get("start", ""), "fin": v["end"],
                               "valeur": v["val"], "depot": v["accn"],
                               "formulaire": v["form"], "depose_le": v["filed"]})
    except Exception as erreur:
        echecs.append((nom, f"{type(erreur).__name__} : {erreur}"))
    time.sleep(0.15)
    if i % 20 == 0:
        print(" ", i, "/", len(entreprises))

actions = pd.DataFrame(lignes)
actions.to_csv(FICHIER, index=False, encoding="utf-8")
print()
print(len(actions), "declarations |", actions.cik.nunique(), "entreprises |", len(echecs), "echecs")
for nom, message in echecs:
    print(" -", nom, ":", message[:70])

  20 / 112
  40 / 112
  60 / 112
  80 / 112
  100 / 112

8513 declarations | 112 entreprises | 0 echecs


### correction 

In [49]:
SUBSTITUTS = [
    ("us-gaap", "CommonStockSharesOutstanding", "shares", "actions_bilan"),
    ("us-gaap", "WeightedAverageNumberOfSharesOutstandingBasic", "shares", "actions_moyenne_ponderee"),
]

avec_actions = set(actions.loc[actions.notion == "actions", "cik"])
manquantes = sorted(set(entreprises.cik) - avec_actions)
print(len(manquantes), "entreprises a completer :", manquantes)

complement = []
for cik in manquantes:
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    facts = requests.get(url, headers=ENTETE, timeout=60).json()["facts"]
    nom = entreprises.loc[entreprises.cik == cik, "nom"].iloc[0]
    for section, etiquette, unite, notion in SUBSTITUTS:
        bloc = facts.get(section, {}).get(etiquette)
        if bloc is None:
            continue
        for v in bloc["units"].get(unite, []):
            complement.append({"cik": cik, "nom": nom, "notion": notion,
                               "debut": v.get("start", ""), "fin": v["end"],
                               "valeur": v["val"], "depot": v["accn"],
                               "formulaire": v["form"], "depose_le": v["filed"]})
    time.sleep(0.15)

actions = pd.concat([actions, pd.DataFrame(complement)], ignore_index=True)

ambigus = (actions[actions.notion == "actions_moyenne_ponderee"]
           .groupby(["cik", "debut", "fin", "depot"])["valeur"].nunique())
if (ambigus > 1).any():
    raise ValueError("Deux valeurs distinctes pour une meme periode et un meme depot")

actions.to_csv(FICHIER, index=False, encoding="utf-8")
print(len(complement), "lignes ajoutees")
print(actions.groupby("notion").cik.nunique().to_string())

3 entreprises a completer : ['0001326801', '0001571996', '0001652044']
474 lignes ajoutees
notion
actions                     109
actions_bilan                 2
actions_moyenne_ponderee      3
flottant_usd                112


In [50]:
import hashlib
import json
from datetime import datetime, timezone

def empreinte(chemin):
    return hashlib.sha256(chemin.read_bytes()).hexdigest()

def resume(chemin):
    d = pd.read_csv(chemin, usecols=["date"])
    return {"fichier": chemin.name, "lignes": len(d),
            "debut": d["date"].iloc[0][:10], "fin": d["date"].iloc[-1][:10],
            "sha256": empreinte(chemin)}

manifeste = {
    "produit_le": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "source_prix": "Yahoo Finance, via la bibliotheque yfinance",
    "source_actions": "SEC, interface XBRL companyfacts",
    "prix": [resume(f) for f in sorted(DOSSIER.glob("*.csv"))],
    "benchmarks": [resume(f) for f in sorted(BENCH.glob("*.csv"))],
    "calendrier": {"fichier": CALENDRIER.name, "seances": len(seances),
                   "sha256": empreinte(CALENDRIER)},
    "actions": {"fichier": FICHIER.name, "lignes": len(actions),
                "entreprises": int(actions.cik.nunique()),
                "sha256": empreinte(FICHIER)},
    "limites": [
        "Yahoo Finance n est pas une source officielle ; ses prix ajustes sont recalcules retroactivement a chaque dividende et a chaque division.",
        "BRK.B est interroge sous le symbole BRK-B ; la correspondance figure dans la colonne symbole_yahoo.",
        "Meta Platforms ne publie pas de nombre d actions a une date dans cette interface ; seule une moyenne ponderee est disponible.",
        "Les nombres d actions sont declares sans retraitement des divisions, contrairement aux prix.",
    ],
}

chemin = RACINE / "data" / "raw" / "prix_manifest.json"
chemin.write_text(json.dumps(manifeste, indent=2, ensure_ascii=False), encoding="utf-8")
print(len(manifeste["prix"]), "fichiers de prix |", len(manifeste["benchmarks"]), "benchmarks")

113 fichiers de prix | 5 benchmarks


In [ ]:
for fichier in sorted(DOSSIER.glob("*.csv")):
    d = pd.read_csv(fichier)
    d.to_csv(fichier.with_suffix(".csv.gz"), index=False, compression="gzip")
    fichier.unlink()

for fichier in sorted(BENCH.glob("*.csv")):
    d = pd.read_csv(fichier)
    d.to_csv(fichier.with_suffix(".csv.gz"), index=False, compression="gzip")
    fichier.unlink()

print("compresse")